In [1]:
# STEP 1: Overview — Script Description
# -----------------------------------
# This script updates longitude values in ESP-r model configuration files.
#
# Workflow:
# 1. Load the EPC dataset (CSV) containing BUILDING_REFERENCE_NUMBER and LONGITUDE.
# 2. Loop through each building folder in the baseline_models directory.
# 3. For each building:
#    - Match the folder name to BUILDING_REFERENCE_NUMBER in the dataset.
#    - Retrieve the corresponding LONGITUDE value.
#    - Search for a "cfg/longitude.txt" file inside any model subfolder.
#    - If found, replace all instances of "LONG" with the actual longitude value.
#    - Save the updated file.
# 4. Skip buildings where:
#    - No matching EPC entry exists
#    - No longitude.txt file is found
#
# Notes:
# - The script is robust to multiple archetype folder names.
# - It will not crash if files are missing.
# - It prints progress so you can track updates.

print("Step 1 complete: Overview loaded")

Step 1 complete: Overview loaded


In [ ]:
# STEP 2: Import Libraries and Define Paths
# ----------------------------------------

import os
import pandas as pd

# File paths
epc_file = "/Users/jakegehrung/Desktop/data_raw/data_processed/fintry_epc_NPV_IRR_update.csv"
baseline_models_path = "/Users/jakegehrung/Desktop/data_raw/baseline_models"

print("Step 2 complete: Libraries imported and paths set")

Step 2 complete: Libraries imported and paths set


In [3]:
# STEP 3: Load EPC Dataset and Prepare Lookup Dictionary
# -----------------------------------------------------

# Load CSV
df = pd.read_csv(epc_file)

# Ensure correct data types
df["BUILDING_REFERENCE_NUMBER"] = df["BUILDING_REFERENCE_NUMBER"].astype(str)

# Create lookup dictionary: {building_id: longitude}
longitude_lookup = dict(zip(df["BUILDING_REFERENCE_NUMBER"], df["LONGITUDE"]))

print(f"Step 3 complete: Loaded {len(longitude_lookup)} building entries")

Step 3 complete: Loaded 117 building entries


In [4]:
# STEP 4: Define Function to Update longitude.txt
# ----------------------------------------------

def update_longitude_file(file_path, longitude_value):
    try:
        with open(file_path, "r") as f:
            content = f.read()
        
        # Replace all instances of 'LONG'
        updated_content = content.replace("LONG", str(longitude_value))
        
        with open(file_path, "w") as f:
            f.write(updated_content)
        
        return True
    
    except Exception as e:
        print(f"Error updating {file_path}: {e}")
        return False

print("Step 4 complete: Update function ready")

Step 4 complete: Update function ready


In [5]:
# STEP 5: Iterate Through Building Folders and Apply Updates
# ---------------------------------------------------------

updated_count = 0
skipped_count = 0

# Loop through building directories
for building_id in os.listdir(baseline_models_path):
    building_path = os.path.join(baseline_models_path, building_id)
    
    # Skip if not a directory
    if not os.path.isdir(building_path):
        continue
    
    # Check if building exists in EPC dataset
    if building_id not in longitude_lookup:
        skipped_count += 1
        continue
    
    longitude_value = longitude_lookup[building_id]
    
    # Loop through archetype/model folders
    for subfolder in os.listdir(building_path):
        subfolder_path = os.path.join(building_path, subfolder)
        
        if not os.path.isdir(subfolder_path):
            continue
        
        cfg_path = os.path.join(subfolder_path, "cfg")
        longitude_file = os.path.join(cfg_path, "longitude.txt")
        
        # Check if longitude.txt exists
        if os.path.exists(longitude_file):
            success = update_longitude_file(longitude_file, longitude_value)
            
            if success:
                updated_count += 1
                print(f"Updated: {longitude_file}")
            else:
                skipped_count += 1

print("Step 5 complete")
print(f"Files updated: {updated_count}")
print(f"Files skipped: {skipped_count}")

Updated: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001664929/SemiDetached_2F/cfg/longitude.txt
Step 5 complete
Files updated: 1
Files skipped: 0


In [6]:
# STEP 6: (Optional) Verify a Sample File
# --------------------------------------

# Replace with a real building ID to check manually
sample_building = list(longitude_lookup.keys())[0]

sample_path = os.path.join(
    baseline_models_path,
    sample_building
)

print(f"Sample building folder: {sample_path}")
print("Manually inspect longitude.txt to confirm replacement worked.")

Sample building folder: /Users/jakegehrung/Desktop/data_raw/baseline_models/1001829067
Manually inspect longitude.txt to confirm replacement worked.
